# Results for model: mediatek/breeze-7b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from typing import List

DATA_PATH = './jane_street_data/train.parquet'
TOP_QUANTILE = 0.15

def calculate_top_quantile(data: pl.DataFrame, feature: str, target: str, batch_size: int) -> pl.Series:
    batches = data.with_columns([pl.col(feature).rolling(window=batch_size, min_periods=1, center=False).quantile(TOP_QUANTILE).alias(f"{feature}_top_quantile")]).groupby("date_id").agg({feature: [pl.col(feature), "date_id"], target: pl.mean(), f"{feature}_top_quantile": pl.first()}).select([feature, target, f"{feature}_top_quantile"])
    return pl.Series(batches[f"{feature}_top_quantile"])

def main():
    data = pl.read_parquet(DATA_PATH)
    feature_00 = "feature_00"
    target = "responder_6"
    batch_size = 100

    top_quantile = calculate_top_quantile(data, feature_00, target, batch_size)

    dtrain = data.select([feature_00, target, pl.lit(True).alias("is_train")])
    dtrain = dtrain.filter(pl.col("is_train") == True)

    xgb_params = {
        "objective": "reg:squarederror",
        "learning_rate": 0.01,
        "max_depth": 6,
        "min_child_weight": 1,
        "subsample": 0.7,
        "colsample_bytree": 0.8,
        "seed": 42
    }

    xgb_model = xgb.XGBRegressor(**xgb_params)
    xgb_model.fit(dtrain[feature_00].values.numpy().astype(float), dtrain[target].values.numpy())

if __name__ == "__main__":
    main()